In [2]:
# 먼저 설치해야 할 패키지들

pip install tkinterdnd2
pip install pip install opencv-python
pip install opencv-python opencv-contrib-python

SyntaxError: invalid syntax (678190651.py, line 3)

In [13]:
import tkinter as tk
from tkinter import filedialog, messagebox, simpledialog
from tkinterdnd2 import TkinterDnD, DND_FILES
import cv2
import os

# Initialize global variables

######## 여기만 편집하기 ########
output_dir = r"C:\Users\liche\KAIST_20 Dropbox\Yi Yunho\CSBD\CSBD_DLC Study\training_videos\Yunho\dataset"
n_chunks = 5
chunk_duration = 60
######## 여기만 편집하기 ########


# 기타 전역변수

input_video_path = ""
tags = []
additional_tag = ""
isQuit = False

# Step 1: Drag & Drop for input video path
def drag_and_drop_window():
    def on_drop(event):
        global input_video_path
        input_video_path = event.data.strip()  # Get the dropped file path

        root.destroy()

    def on_quit():
        global isQuit
        isQuit = True
        root.destroy()

    root = TkinterDnD.Tk()  # Initialize a DND-enabled Tkinter window
    root.title("Drag & Drop Video File")
    root.geometry("400x200")

    label = tk.Label(root, text="Drag and drop your video file here", font=("Arial", 14))
    label.pack(pady=50)

    button_frame = tk.Frame(root)
    button_frame.pack(pady=20)

    submit_button = tk.Button(button_frame, text="Submit", command=root.destroy)
    submit_button.pack(side=tk.LEFT, padx=10)

    quit_button = tk.Button(button_frame, text="Quit", command=on_quit)
    quit_button.pack(side=tk.LEFT, padx=10)

    root.drop_target_register(DND_FILES)
    root.dnd_bind('<<Drop>>', on_drop)

    root.mainloop()

# Step 2: Output directory input
# def output_directory_window():
#     def on_submit():
#         global output_dir
#         output_dir = dir_entry.get().strip()
#         if os.path.isdir(output_dir):
#             messagebox.showinfo("Directory Selected", f"Selected output directory: {output_dir}")
#             root.destroy()
#         else:
#             messagebox.showerror("Invalid Directory", "Please enter a valid directory path.")

#     root = tk.Tk()
#     root.title("Output Directory")
#     root.geometry("400x150")

#     tk.Label(root, text="Enter output directory:", font=("Arial", 12)).pack(pady=10)
#     dir_entry = tk.Entry(root, width=50)
#     dir_entry.pack(pady=10)

#     submit_button = tk.Button(root, text="Submit", command=on_submit)
#     submit_button.pack(pady=10)

#     root.mainloop()

# Step 3: Tag selection
def tags_selection_window():
    global tags
    if not tags:  # Initialize with default values if no previous selection
        tags = ["Select M or C", "Select SA or MA", "Select WH or NW", "Select category", "Select F or T", "", ""]
    global additional_tag

    def on_submit():
        global tags
        tags = [first_var.get(), second_var.get(), third_var.get(), fourth_var.get(), fifth_var.get(), sixth_entry.get()]
        additional_tag = seventh_entry.get()
        if any(tag == "" for tag in tags[:-1]):
            messagebox.showerror("Invalid Input", "Please make a selection for all dropdowns.")
        else:
            root.destroy()

    root = tk.Tk()
    root.title("Tag Selection")
    root.geometry("500x600")

    tk.Label(root, text="Select Tags", font=("Arial", 14)).pack(pady=10)

    # Dropdown options
    first_var = tk.StringVar(value=tags[0])
    second_var = tk.StringVar(value=tags[1])
    third_var = tk.StringVar(value=tags[2])
    fourth_var = tk.StringVar(value=tags[3])
    fifth_var = tk.StringVar(value=tags[4])
    sixth_entry = tk.StringVar(value=tags[5])

    tk.Label(root, text="Color (M/C):").pack()
    tk.OptionMenu(root, first_var, "M", "C").pack()

    tk.Label(root, text="Number of animal (SA/MA):").pack()
    tk.OptionMenu(root, second_var, "SA", "MA").pack()

    tk.Label(root, text="Background color:").pack()
    tk.OptionMenu(root, third_var, "WH", "NW").pack()

    tk.Label(root, text="Experiment:").pack()
    tk.OptionMenu(root, fourth_var, "3C", "DI", "LD", "OF", "EP", "HC", "NO").pack()

    tk.Label(root, text="Head apparatus:").pack()
    tk.OptionMenu(root, fifth_var, "F", "T").pack()

    tk.Label(root, text="Experimenter:").pack()
    sixth_entry = tk.Entry(root, width=50)
    sixth_entry.insert(0, tags[5])  # Set the previous text value
    sixth_entry.pack(pady=10)

    tk.Label(root, text="Additional tag (optional):").pack()
    seventh_entry = tk.Entry(root, width=50)
    seventh_entry.insert(0, additional_tag)
    seventh_entry.pack(pady=10)

    submit_button = tk.Button(root, text="Submit", command=on_submit)
    submit_button.pack(pady=10)

    root.mainloop()


# Step 4: Number of chunks input
# def chunks_input_window():
#     def on_submit():
#         global n_chunks
#         try:
#             n_chunks = int(chunk_entry.get().strip())
#             if n_chunks > 0:
#                 root.destroy()
#             else:
#                 messagebox.showerror("Invalid Input", "Please enter a positive integer.")
#         except ValueError:
#             messagebox.showerror("Invalid Input", "Please enter a valid integer.")

#     root = tk.Tk()
#     root.title("Number of Chunks")
#     root.geometry("300x150")

#     tk.Label(root, text="How many chunks?", font=("Arial", 12)).pack(pady=10)
#     chunk_entry = tk.Entry(root, width=20)
#     chunk_entry.pack(pady=10)

#     submit_button = tk.Button(root, text="Submit", command=on_submit)
#     submit_button.pack(pady=10)

#     root.mainloop()

# Step 5: Crop & Split
def VideoChopperAndCrop(input_video_path: str, output_dir: str, tags: list = [], chunk_duration: int = 60, n_chunks: int = 5):
    """
    Splits the input video into chunks of given duration and allows the user to crop a region of interest (ROI).
    The output file name includes tags, video number, and end_tag in the format: tags_video_number_endtag
    
    Args:
        input_video_path (str): Path to the input video file.
        output_dir (str): Directory to save the cropped and chunked videos.
        tags (list of str): Tags to append to the chunk names (e.g., ["tag1", "tag2"]).
        chunk_duration (int): Duration of each chunk in seconds (default: 60).
        n_chunks (int): Number of chunks to generate (default: 5).
    """
    
    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Get the file extension of the input video
    _, extension = os.path.splitext(input_video_path)
        
    # Open the input video
    video = cv2.VideoCapture(input_video_path)
    
    if not video.isOpened():
        print(f"Error: Could not open video {input_video_path}")
        return
    
    # Get video properties
    fps = int(video.get(cv2.CAP_PROP_FPS))
    frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps  # Total duration of the video in seconds

    print(f"Video loaded: {input_video_path}")
    print(f"Total Duration: {duration:.2f}s, FPS: {fps}, Resolution: {frame_width}x{frame_height}")
    
    chunk_frames = chunk_duration * fps  # Convert time to frame count

    # Initialize variables
    frame_idx = 0
    chunk_idx = 0
    file_idx = 0
    
    # Read the first frame to allow user to select ROI
    ret, frame = video.read()
    if not ret:
        print("Error: Could not read the first frame.")
        return

    # Allow the user to select ROI, if nothing is selected, the entire frame is considered the ROI
    roi = cv2.selectROI("Select ROI", frame, fromCenter=False, showCrosshair=True)
    cv2.destroyWindow("Select ROI")

    # If no ROI selected (user pressed Enter without selecting), use the entire frame
    if roi == (0, 0, 0, 0):
        print("No ROI selected. Using the entire frame as ROI.")
        roi = (0, 0, frame_width, frame_height)  # Set the entire frame as ROI

    x, y, w, h = roi  # Extract ROI coordinates (x, y, width, height)
    cropped_frame = frame[y:y+h, x:x+w]
    
    file_list = [os.path.splitext(file)[0] for file in os.listdir(output_dir)] # 파일 목록 확인. 이름 겹치지 않게
    
    # Initialize VideoWriter
    while chunk_idx < n_chunks:
        ret, frame = video.read()
        if not ret:  # End of video
            break
        
        # Start a new chunk when the current chunk duration is reached
        if frame_idx % chunk_frames == 0:
            if frame_idx > 0:  # Release the previous writer
                writer.release()
                chunk_idx += 1

            if not (chunk_idx < n_chunks):
                break


            # Determine the codec based on the file extension
            if extension == ".avi":
                fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Codec for avi format
            elif extension == ".mp4":
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for mp4 format
            else:
                print("Unknown file type; codec error")
                return

            
            

            # IMPLEMENT: 파일이름이 겹칠 때 덮어쓰기가 되지 않도록, 겹치지 않는 3자릿수 인덱싱을 사용. (확장자와 관계없이)
            # 전략: 
            # 1. working directory에 이미 설정한 filename이 포함되어 있는지 확인
            # 2. 파일 이름이 겹칠 경우, idx를 1 증가시켜보기.
            # 3. 1을 반복.

            concatenated_tags = "_".join(filter(lambda tag: tag.strip() != "", tags))
            processed_file_name = f"{concatenated_tags}_{str(file_idx).zfill(3)}"

            while processed_file_name in file_list:
                # Build the chunk file name with tags, video number, and end_tag
                tag_part = "_".join(tags) if tags else ""
                file_idx += 1
                processed_file_name = f"{concatenated_tags}_{str(file_idx).zfill(3)}"

                
            
            
            chunk_file = os.path.join(output_dir, processed_file_name + extension)
            
            writer = cv2.VideoWriter(chunk_file, fourcc, fps, (w, h))
            
            file_idx += 1

            print(f"Started new chunk: {chunk_file}")

        # Crop the frame using the selected ROI
        
        
        # Write the cropped frame to the current chunk
        writer.write(cropped_frame)
        frame_idx += 1

    # Release resources
    writer.release()
    video.release()
    print(f"Video chopping and cropping completed! {chunk_idx} chunks were created.")


# Main execution
if __name__ == "__main__":

    while not isQuit:
        drag_and_drop_window()  # Step 1
        if isQuit:
            break
        #output_directory_window()  # Step 2
        tags_selection_window()  # Step 3
        #chunks_input_window()  # Step 4
        
        VideoChopperAndCrop(input_video_path[1:-1], output_dir, tags, chunk_duration, n_chunks)
        
        # Debugging Output
        print("Input Video Path:", input_video_path)
        print("Output Directory:", output_dir)
        print("Tags:", tags)
        print("Number of Chunks:", n_chunks)



Video loaded: C:/Users/liche/KAIST_20 Dropbox/Yi Yunho/CSBD/CSBD_DLC Study/training_videos/Yunho/Original videos/8_5_20221101_161934.avi
Total Duration: 498.48s, FPS: 29, Resolution: 640x480
No ROI selected. Using the entire frame as ROI.
Started new chunk: C:\Users\liche\KAIST_20 Dropbox\Yi Yunho\CSBD\CSBD_DLC Study\training_videos\Yunho\dataset\M_SA_NW_EP_F_Yunho_005.avi
Started new chunk: C:\Users\liche\KAIST_20 Dropbox\Yi Yunho\CSBD\CSBD_DLC Study\training_videos\Yunho\dataset\M_SA_NW_EP_F_Yunho_006.avi
Started new chunk: C:\Users\liche\KAIST_20 Dropbox\Yi Yunho\CSBD\CSBD_DLC Study\training_videos\Yunho\dataset\M_SA_NW_EP_F_Yunho_007.avi
Started new chunk: C:\Users\liche\KAIST_20 Dropbox\Yi Yunho\CSBD\CSBD_DLC Study\training_videos\Yunho\dataset\M_SA_NW_EP_F_Yunho_008.avi
Started new chunk: C:\Users\liche\KAIST_20 Dropbox\Yi Yunho\CSBD\CSBD_DLC Study\training_videos\Yunho\dataset\M_SA_NW_EP_F_Yunho_009.avi
Video chopping and cropping completed! 5 chunks were created.
Input Video Pa

In [13]:
input_video_path[1:-1]

'C:/Users/liche/KAIST_20 Dropbox/Yi Yunho/CSBD/CSBD_DLC Study/training_videos/Yunho/Original videos/240215_AF;C_txf_coh2_OF_100lx20240215_170737.avi'